In [1]:
import sys
print(sys.executable)

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports successful")

c:\Users\sehba\OneDrive - Asia Pacific University of Technology And Innovation (APU)\Documents\fraud-detector-fyp\backend\venv\Scripts\python.exe
All imports successful


In [2]:
train_df = pd.read_csv("../data/processed/train.csv", index_col="original_index")
val_df = pd.read_csv("../data/processed/val.csv", index_col="original_index")
test_df = pd.read_csv("../data/processed/test.csv", index_col="original_index")

print("Shapes:")
print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

print("\nClass balance:")
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    fraud_count = d["fraudulent"].sum()
    total = len(d)
    print(f"{name}: {fraud_count} fraudulent / {total} total ({fraud_count/total*100:.2f}%)")

Shapes:
Train: (12516, 3)
Val:   (2682, 3)
Test:  (2682, 3)

Class balance:
Train: 606 fraudulent / 12516 total (4.84%)
Val: 130 fraudulent / 2682 total (4.85%)
Test: 130 fraudulent / 2682 total (4.85%)


In [3]:
print("\nNulls in clean_text:")
print("Train:", train_df["clean_text"].isnull().sum())
print("Val:  ", val_df["clean_text"].isnull().sum())
print("Test: ", test_df["clean_text"].isnull().sum())

print("\nfraudulent dtype:", train_df["fraudulent"].dtype)


Nulls in clean_text:
Train: 0
Val:   0
Test:  0

fraudulent dtype: int64


In [4]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)

X_train = vectorizer.fit_transform(train_df["clean_text"])
X_val = vectorizer.transform(val_df["clean_text"])
X_test = vectorizer.transform(test_df["clean_text"])

y_train = train_df["fraudulent"].values
y_val = val_df["fraudulent"].values
y_test = test_df["fraudulent"].values

print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)
print("X_test shape: ", X_test.shape)

X_train shape: (12516, 10000)
X_val shape:   (2682, 10000)
X_test shape:  (2682, 10000)


In [5]:
feature_names = vectorizer.get_feature_names_out()
print("Vocabulary size:", len(feature_names))
print("Sample terms:", feature_names[:20])

Vocabulary size: 10000
Sample terms: ['aa' 'aaa' 'aac' 'aan' 'ab' 'abap' 'abc' 'aberdeen' 'abilities' 'ability'
 'able' 'aboard' 'abreast' 'abroad' 'absence' 'absent' 'absolute'
 'absolutely' 'abstract' 'abuse']


In [6]:
suspicious_terms = ["aabbf", "aacd", "abafa", "abede", "abfceafd"]

for term in suspicious_terms:
    matches = train_df[train_df["clean_text"].str.contains(term, regex=False)]
    print(f"\n--- '{term}' appears in {len(matches)} training postings ---")
    if len(matches) > 0:
        text = matches.iloc[0]["clean_text"]
        idx = text.find(term)
        start = max(0, idx - 80)
        end = min(len(text), idx + 80)
        print("Context:", text[start:end])


--- 'aabbf' appears in 0 training postings ---

--- 'aacd' appears in 0 training postings ---

--- 'abafa' appears in 0 training postings ---

--- 'abede' appears in 0 training postings ---

--- 'abfceafd' appears in 0 training postings ---


In [7]:
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    residue = d["clean_text"].str.contains(r"#[a-z]+_[0-9a-f]+#", regex=True).sum()
    print(f"{name} placeholder residue: {residue}")

Train placeholder residue: 0
Val placeholder residue: 0
Test placeholder residue: 0


In [8]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)

X_train = vectorizer.fit_transform(train_df["clean_text"])
X_val = vectorizer.transform(val_df["clean_text"])
X_test = vectorizer.transform(test_df["clean_text"])

y_train = train_df["fraudulent"].values
y_val = val_df["fraudulent"].values
y_test = test_df["fraudulent"].values

print("X_train shape:", X_train.shape)
print("X_val shape:  ", X_val.shape)
print("X_test shape: ", X_test.shape)

feature_names = vectorizer.get_feature_names_out()
print("\nVocabulary size:", len(feature_names))
print("First 20 alphabetically:", feature_names[:20])

X_train shape: (12516, 10000)
X_val shape:   (2682, 10000)
X_test shape:  (2682, 10000)

Vocabulary size: 10000
First 20 alphabetically: ['aa' 'aaa' 'aac' 'aan' 'ab' 'abap' 'abc' 'aberdeen' 'abilities' 'ability'
 'able' 'aboard' 'abreast' 'abroad' 'absence' 'absent' 'absolute'
 'absolutely' 'abstract' 'abuse']


In [9]:
suspicious_terms_check = ["aabbf", "aacd", "abafa", "abede", "abfceafd"]

still_in_vocab = [t for t in suspicious_terms_check if t in feature_names]

print("Old garbage terms still in vocabulary:", still_in_vocab)

Old garbage terms still in vocabulary: []


In [10]:
smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(f"  Genuine: {(y_train == 0).sum()}, Fraudulent: {(y_train == 1).sum()}")

print("\nAfter SMOTE:")
print(f"  Genuine: {(y_train_res == 0).sum()}, Fraudulent: {(y_train_res == 1).sum()}")

print(f"\nX_train_res shape: {X_train_res.shape}")

Before SMOTE:
  Genuine: 11910, Fraudulent: 606

After SMOTE:
  Genuine: 11910, Fraudulent: 11910

X_train_res shape: (23820, 10000)


In [11]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_res, y_train_res)
print("Logistic Regression trained")

nb = MultinomialNB()
nb.fit(X_train_res, y_train_res)
print("Naive Bayes trained")

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_res, y_train_res)
print("Random Forest trained")

Logistic Regression trained
Naive Bayes trained
Random Forest trained


In [12]:
models = {
    "Logistic Regression": log_reg,
    "Naive Bayes": nb,
    "Random Forest": rf,
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]

    report = classification_report(
        y_val,
        y_pred,
        target_names=["genuine", "fraudulent"],
        output_dict=True,
    )
    auc = roc_auc_score(y_val, y_proba)

    results.append({
        "Model": name,
        "Accuracy": report["accuracy"],
        "Precision (fraud)": report["fraudulent"]["precision"],
        "Recall (fraud)": report["fraudulent"]["recall"],
        "F1 (fraud)": report["fraudulent"]["f1-score"],
        "ROC-AUC": auc,
    })

    print(f"=== {name} ===")
    print(classification_report(
        y_val,
        y_pred,
        target_names=["genuine", "fraudulent"]
    ))
    print("ROC-AUC:", round(auc, 4))
    print("Confusion Matrix (rows=actual, cols=predicted):")
    print(confusion_matrix(y_val, y_pred))
    print()

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

=== Logistic Regression ===
              precision    recall  f1-score   support

     genuine       0.99      0.99      0.99      2552
  fraudulent       0.75      0.84      0.79       130

    accuracy                           0.98      2682
   macro avg       0.87      0.91      0.89      2682
weighted avg       0.98      0.98      0.98      2682

ROC-AUC: 0.9687
Confusion Matrix (rows=actual, cols=predicted):
[[2515   37]
 [  21  109]]

=== Naive Bayes ===
              precision    recall  f1-score   support

     genuine       0.99      0.95      0.97      2552
  fraudulent       0.45      0.83      0.58       130

    accuracy                           0.94      2682
   macro avg       0.72      0.89      0.78      2682
weighted avg       0.96      0.94      0.95      2682

ROC-AUC: 0.9565
Confusion Matrix (rows=actual, cols=predicted):
[[2420  132]
 [  22  108]]

=== Random Forest ===
              precision    recall  f1-score   support

     genuine       0.98      1.00    

In [13]:
thresholds_to_test = [0.5, 0.3, 0.2, 0.1]
threshold_results = []

for name, model in models.items():
    y_proba = model.predict_proba(X_val)[:, 1]

    for t in thresholds_to_test:
        y_pred_t = (y_proba >= t).astype(int)

        report = classification_report(
            y_val,
            y_pred_t,
            target_names=["genuine", "fraudulent"],
            output_dict=True,
            zero_division=0
        )

        threshold_results.append({
            "Model": name,
            "Threshold": t,
            "Precision (fraud)": round(report["fraudulent"]["precision"], 3),
            "Recall (fraud)": round(report["fraudulent"]["recall"], 3),
            "F1 (fraud)": round(report["fraudulent"]["f1-score"], 3),
        })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

              Model  Threshold  Precision (fraud)  Recall (fraud)  F1 (fraud)
Logistic Regression        0.5              0.747           0.838       0.790
Logistic Regression        0.3              0.545           0.923       0.686
Logistic Regression        0.2              0.389           0.938       0.550
Logistic Regression        0.1              0.200           0.954       0.331
        Naive Bayes        0.5              0.450           0.831       0.584
        Naive Bayes        0.3              0.336           0.885       0.487
        Naive Bayes        0.2              0.263           0.908       0.408
        Naive Bayes        0.1              0.185           0.938       0.310
      Random Forest        0.5              0.988           0.638       0.776
      Random Forest        0.3              0.812           0.831       0.821
      Random Forest        0.2              0.529           0.915       0.670
      Random Forest        0.1              0.214           0.96

In [14]:
import joblib
import json
import os

os.makedirs("../saved_models", exist_ok=True)

joblib.dump(rf, "../saved_models/baseline_random_forest.joblib")
joblib.dump(vectorizer, "../saved_models/tfidf_vectorizer.joblib")

metadata = {
    "model_type": "RandomForestClassifier",
    "n_estimators": 200,
    "decision_threshold": 0.3,
    "validation_metrics": {
        "precision_fraud": 0.812,
        "recall_fraud": 0.831,
        "f1_fraud": 0.821,
    },
    "vectorizer_config": "TfidfVectorizer(stop_words='english', max_features=10000)",
    "trained_on": "SMOTE-resampled training set (23820 rows, balanced 1:1)",
}

with open("../saved_models/baseline_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved: baseline_random_forest.joblib, tfidf_vectorizer.joblib, baseline_metadata.json")

Saved: baseline_random_forest.joblib, tfidf_vectorizer.joblib, baseline_metadata.json


In [15]:
rf_reloaded = joblib.load("../saved_models/baseline_random_forest.joblib")
vectorizer_reloaded = joblib.load("../saved_models/tfidf_vectorizer.joblib")

original_preds = rf.predict(X_val[:5])
reloaded_preds = rf_reloaded.predict(X_val[:5])

print("Reloaded model predictions match original:",
      (original_preds == reloaded_preds).all())

with open("../saved_models/baseline_metadata.json") as f:
    print(json.load(f))

Reloaded model predictions match original: True
{'model_type': 'RandomForestClassifier', 'n_estimators': 200, 'decision_threshold': 0.3, 'validation_metrics': {'precision_fraud': 0.812, 'recall_fraud': 0.831, 'f1_fraud': 0.821}, 'vectorizer_config': "TfidfVectorizer(stop_words='english', max_features=10000)", 'trained_on': 'SMOTE-resampled training set (23820 rows, balanced 1:1)'}
